In [6]:
import pandas as pd

In [ ]:
tags_df= pd.read_csv("C:/Users/Daksh/Desktop/movie-recommendation/data/development/ml-latest/tags.csv")
links_df= pd.read_csv("C:/Users/Daksh/Desktop/movie-recommendation/data/development/ml-latest/links.csv")
ratings_df= pd.read_csv("C:/Users/Daksh/Desktop/movie-recommendation/data/development/ml-latest/ratings.csv")
movies_df= pd.read_csv("C:/Users/Daksh/Desktop/movie-recommendation/data/development/ml-latest/movies.csv")
genome_tags_df= pd.read_csv("C:/Users/Daksh/Desktop/movie-recommendation/data/development/ml-latest/genome-tags.csv")
genome_scores_df= pd.read_csv("C:/Users/Daksh/Desktop/movie-recommendation/data/development/ml-latest/genome-scores.csv")

In [ ]:
tags_df.head()

,userId,movieId,tag,timestamp
0,10,260,good vs evil,1430666558
1,10,260,Harrison Ford,1430666505
2,10,260,sci-fi,1430666538
3,14,1221,Al Pacino,1311600756
4,14,1221,mafia,1311600746


In [ ]:
movies_df.head()

In [ ]:
ratings_df.head()

In [ ]:
links_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [ ]:
import requests
import json
import time  # For rate limiting

# Step 1: Compute average ratings
avg_ratings = ratings_df.groupby('movieId')['rating'].mean().reset_index(name='avg_rating')

# Step 2: Aggregate user tags (unique per movie)
# Convert 'tag' column to string to handle potential non-string types
tags_df['tag'] = tags_df['tag'].astype(str)
user_tags = tags_df.groupby('movieId')['tag'].apply(lambda x: '|'.join(x.unique())).reset_index(name='user_tags')

# Step 3: Create genome relevances as dict per movie
genome_merged = genome_scores_df.merge(genome_tags_df, on='tagId')
genome_dicts = genome_merged.groupby('movieId').apply(
    lambda x: json.dumps(dict(zip(x['tag'], x['relevance'])))
).reset_index(name='genome_relevances')

# Step 4: Merge base data
enriched_df = movies_df.merge(links_df, on='movieId', how='left')
enriched_df = enriched_df.merge(avg_ratings, on='movieId', how='left')
enriched_df = enriched_df.merge(user_tags, on='movieId', how='left')
enriched_df = enriched_df.merge(genome_dicts, on='movieId', how='left')

# Fill NaNs
enriched_df['avg_rating'] = enriched_df['avg_rating'].fillna(0)
enriched_df['user_tags'] = enriched_df['user_tags'].fillna('')
enriched_df['genome_relevances'] = enriched_df['genome_relevances'].fillna('{}')

# Step 5: Fetch TMDB data (batch to avoid rate limits)
TMDB_API_KEY = '549d74d021bd35eac9d680ec3ec9aacf'  # Replace with your key
BASE_URL = 'https://api.themoviedb.org/3/movie/'
IMAGE_BASE = 'https://image.tmdb.org/t/p/w500/'

def fetch_tmdb_details(tmdb_id):
    if pd.isna(tmdb_id):
        return None, None, None, None
    url = f"{BASE_URL}{int(tmdb_id)}?api_key={TMDB_API_KEY}&append_to_response=credits"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            overview = data.get('overview', '')
            poster_path = data.get('poster_path', '')
            poster_url = f"{IMAGE_BASE}{poster_path}" if poster_path else ''

            # Director: First director from crew
            directors = [crew['name'] for crew in data['credits']['crew'] if crew['job'] == 'Director']
            director = '|'.join(directors[:3])  # Top 3 if multiple

            # Actors: Top 5 cast
            actors = [cast['name'] for cast in data['credits']['cast'][:5]]
            actors_str = '|'.join(actors)

            # print(director, actors_str, overview, poster_url)
            return director, actors_str, overview, poster_url
        else:
            return None, None, None, None
    except Exception:
        return None, None, None, None

# Add TMDB columns
enriched_df['director'] = None
enriched_df['actors'] = None
enriched_df['overview'] = None
enriched_df['poster_url'] = None

# Fetch in batches (e.g., every 40 requests, sleep 1 sec for rate limit)
for idx, row in enriched_df.iterrows():
    director, actors, overview, poster_url = fetch_tmdb_details(row['tmdbId'])
    enriched_df.at[idx, 'director'] = director or ''
    enriched_df.at[idx, 'actors'] = actors or ''
    enriched_df.at[idx, 'overview'] = overview or ''
    enriched_df.at[idx, 'poster_url'] = poster_url or ''

    if idx % 40 == 0 and idx > 0:
        time.sleep(1)  # Rate limit buffer

# Save to CSV
enriched_df.to_csv('enriched_movies.csv', index=False)
print("Enriched CSV created: enriched_movies.csv")